# A Movie Recommendation Service
### Source: https://www.codementor.io/@jadianes/building-a-recommender-with-apache-spark-python-example-app-part1-du1083qbw

#### Create a SparkContext configured for local mode

In [1]:
import pyspark
sc = pyspark.SparkContext('local[*]')

#### File download
Small: 100,000 ratings and 3,600 tag applications applied to 9,000 movies by 600 users. Last updated 9/2018.  
Full: approximately 33,000,000 ratings and 2,000,000 tag applications applied to 86,000 movies by 330,975 users. Includes tag genome data with 14 million relevance scores across 1,100 tags. Last updated 9/2018.

In [2]:
# Using complete dataset here
complete_dataset_url = 'http://files.grouplens.org/datasets/movielens/ml-latest.zip'

#### Download location(s)

In [3]:
import os
datasets_path = os.path.join('/home/jovyan', 'work')
complete_dataset_path = os.path.join(datasets_path, 'ml-latest.zip')

#### Getting file(s)

In [ ]:
import urllib.request
complete_f = urllib.request.urlretrieve (complete_dataset_url, complete_dataset_path)

#### Extracting file(s)

In [ ]:
import zipfile

with zipfile.ZipFile(complete_dataset_path, "r") as z:
    z.extractall(datasets_path)

## Loading and parsing datasets
Now we are ready to read in each of the files and create an RDD consisting of parsed lines. 

Each line in the ratings dataset (ratings.csv) is formatted as: 
+ userId,movieId,rating,timestamp 

Each line in the movies (movies.csv) dataset is formatted as:
+ movieId,title,genres 

The format of these files is uniform and simple, so we can use Python split() to parse their lines once they are loaded into RDDs. Parsing the movies and ratings files yields two RDDs: 
+ For each line in the ratings dataset, we create a tuple of (UserID, MovieID, Rating). We drop the timestamp because we do not need it for this recommender.
+ For each line in the movies dataset, we create a tuple of (MovieID, Title). We drop the genres because we do not use them for this recommender.

#### ratings.csv

In [4]:
complete_ratings_file = os.path.join(datasets_path, 'ml-latest', 'ratings.csv')
complete_ratings_raw_data = sc.textFile(complete_ratings_file)
complete_ratings_raw_data_header = complete_ratings_raw_data.take(1)[0]
# Parse
complete_ratings_data = complete_ratings_raw_data.filter(lambda line: line!=complete_ratings_raw_data_header)\
    .map(lambda line: line.split(",")).map(lambda tokens: (int(tokens[0]),int(tokens[1]),float(tokens[2]))).cache()

print ('There are {} recommendations in the complete dataset'.format(complete_ratings_data.count()))
complete_ratings_data.take(3)

There are 33832162 recommendations in the complete dataset


[(1, 1, 4.0), (1, 110, 4.0), (1, 158, 4.0)]

#### movies.csv

In [5]:
# Load the small dataset file
complete_movies_file = os.path.join(datasets_path, 'ml-latest', 'movies.csv')
complete_movies_raw_data = sc.textFile(complete_movies_file)
complete_movies_raw_data_header = complete_movies_raw_data.take(1)[0]

# Parse
complete_movies_data = complete_movies_raw_data.filter(lambda line: line!=complete_movies_raw_data_header)\
    .map(lambda line: line.split(",")).map(lambda tokens: (int(tokens[0]),tokens[1],tokens[2])).cache()

complete_movies_titles = complete_movies_data.map(lambda x: (int(x[0]),x[1]))
print ('There are {} movies in the complete dataset'.format(complete_movies_titles.count()))
complete_movies_data.take(3)

There are 86537 movies in the complete dataset


[(1, 'Toy Story (1995)', 'Adventure|Animation|Children|Comedy|Fantasy'),
 (2, 'Jumanji (1995)', 'Adventure|Children|Fantasy'),
 (3, 'Grumpier Old Men (1995)', 'Comedy|Romance')]

## Collaborative Filtering
In Collaborative filtering we make predictions (filtering) about the interests of a user by collecting preferences or taste information from many users (collaborating). The underlying assumption is that if a user A has the same opinion as a user B on an issue, A is more likely to have B's opinion on a different issue x than to have the opinion on x of a user chosen randomly. 

At first, people rate different items (like videos, images, games). Then, the system makes predictions about a user's rating for an item not rated yet. The new predictions are built upon the existing ratings of other users with similar ratings with the active user. In the image, the system predicts that the user will not like the video.

Spark MLlib library for Machine Learning provides a Collaborative Filtering implementation by using Alternating Least Squares. The implementation in MLlib has the following parameters:

+ numBlocks is the number of blocks used to parallelize computation (set to -1 to auto-configure).
+ rank is the number of latent factors in the model.
+ iterations is the number of iterations to run.
+ lambda specifies the regularization parameter in ALS.
+ implicitPrefs specifies whether to use the explicit feedback ALS variant or one adapted for implicit feedback data.
+ alpha is a parameter applicable to the implicit feedback variant of ALS that governs the baseline confidence in preference observations.

#### Selecting ALS parameters using the small dataset
In order to determine the best ALS parameters, we will use the small dataset. We need first to split it into train, validation, and test datasets.

In [6]:
# source uses see=0L, which is the previous version of python (2.x)
# 0L should be written as 0 from now on
training_RDD, validation_RDD, test_RDD = complete_ratings_data.randomSplit([6, 2, 2], seed=0)
validation_for_predict_RDD = validation_RDD.map(lambda x: (x[0], x[1]))
test_for_predict_RDD = test_RDD.map(lambda x: (x[0], x[1]))

#### Training phase

In [7]:
from pyspark.mllib.recommendation import ALS
import math

seed = 5
iterations = 10
regularization_parameter = 0.1
ranks = [4, 8, 12]
errors = [0, 0, 0]
err = 0
tolerance = 0.02

min_error = float('inf')
best_rank = -1
best_iteration = -1

for rank in ranks:
    model = ALS.train(training_RDD, rank, seed=seed, iterations=iterations,
                      lambda_=regularization_parameter)
    predictions = model.predictAll(validation_for_predict_RDD).map(lambda r: ((r[0], r[1]), r[2]))
    rates_and_preds = validation_RDD.map(lambda r: ((int(r[0]), int(r[1])), float(r[2]))).join(predictions)
    error = math.sqrt(rates_and_preds.map(lambda r: (r[1][0] - r[1][1])**2).mean())
    errors[err] = error
    err += 1
    print ('For rank {} the RMSE is {}'.format(rank, error))
    if error < min_error:
        min_error = error
        best_rank = rank

print ('The best model was trained with rank {}'.format(best_rank))

For rank 4 the RMSE is 0.8264132512469051
For rank 8 the RMSE is 0.8125157602875798
For rank 12 the RMSE is 0.8099430227420116
The best model was trained with rank 12


## Using the complete dataset to build the final model
Due to the limitations of virtual machine, we keep using the small dataset instead of complete dataset

We need first to split it into training and test datasets.

In [8]:
training_RDD, test_RDD = complete_ratings_data.randomSplit([7, 3], seed=0)

complete_model = ALS.train(training_RDD, best_rank, seed=seed, \
                           iterations=iterations, lambda_=regularization_parameter)

Now we test on our testing set.

In [9]:
test_for_predict_RDD = test_RDD.map(lambda x: (x[0], x[1]))

predictions = complete_model.predictAll(test_for_predict_RDD).map(lambda r: ((r[0], r[1]), r[2]))
rates_and_preds = test_RDD.map(lambda r: ((int(r[0]), int(r[1])), float(r[2]))).join(predictions)
error = math.sqrt(rates_and_preds.map(lambda r: (r[1][0] - r[1][1])**2).mean())

print ('For testing data the RMSE is {}'.format(error))

For testing data the RMSE is 0.8067765764657743


## How to make recommendations
Although we aim at building an online movie recommender, now that we know how to have our recommender model ready, we can give it a try providing some movie recommendations. This will help us coding the recommending engine later on when building the web service, and will explain how to use the model in any other circumstances.

When using collaborative filtering, getting recommendations is not as simple as predicting for the new entries using a previously generated model. Instead, we need to train again the model but including the new user preferences in order to compare them with other users in the dataset. That is, the recommender needs to be trained every time we have new user ratings (although a single model can be used by multiple users of course!). This makes the process expensive, and it is one of the reasons why scalability is a problem (and Spark a solution!). Once we have our model trained, we can reuse it to obtain top recomendations for a given user or an individual rating for a particular movie. These are less costly operations than training the model itself.

Another thing we want to do, is give recommendations of movies with a certain minimum number of ratings. For that, we need to count the number of ratings per movie.

In [10]:
def get_counts_and_averages(ID_and_ratings_tuple):
    nratings = len(ID_and_ratings_tuple[1])
    return ID_and_ratings_tuple[0], (nratings, float(sum(x for x in ID_and_ratings_tuple[1]))/nratings)

movie_ID_with_ratings_RDD = (complete_ratings_data.map(lambda x: (x[1], x[2])).groupByKey())
movie_ID_with_avg_ratings_RDD = movie_ID_with_ratings_RDD.map(get_counts_and_averages)
movie_rating_counts_RDD = movie_ID_with_avg_ratings_RDD.map(lambda x: (x[0], x[1][0]))

### Adding new user ratings
Now we need to rate some movies for the new user. We will put them in a new RDD and we will use the user ID 0, that is not assigned in the MovieLens dataset. Check the dataset movies file for ID to Tittle assignment (so you know what movies are you actually rating).

In [11]:
new_user_ID = 0

# The format of each line is (userID, movieID, rating)

# ###################################################
# Keep the userID, but Replace movieID, rating, title
# ###################################################

# Find 10 movies you have watched in the past
# Put your OWN ratings

# new_user_ratings = [
#      (0,260,4), # Star Wars (1977)
#      (0,1,3), # Toy Story (1995)
#      (0,16,3), # Casino (1995)
#      (0,25,4), # Leaving Las Vegas (1995)
#      (0,32,4), # Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
#      (0,335,1), # Flintstones, The (1994)
#      (0,379,1), # Timecop (1994)
#      (0,296,3), # Pulp Fiction (1994)
#      (0,858,5), # Godfather, The (1972)
#      (0,50,4) # Usual Suspects, The (1995)
#     ]

# new_user_ratings = [
#      (0,180045,5), # Molly's Game (2017)
#      (0,185029,1), # A Quiet Place (2018)
#      (0,187541,4), # Incredibles 2 (2018)
#      (0,1,4), # Toy Story (1995)
#      (0,8808,3), # Princess Diaries 2: Royal Engagement, The (2004)
#      (0,46062,3), # High School Musical (2006)
#      (0,45720,3), # Devil Wears Prada, The (2006)
#      (0,50872,5), # Ratatouille (2007)
#      (0,54001,4), # Harry Potter and the Order of the Phoenix (2007)
#      (0,55768,2) # Bee Movie (2007)
#     ]

# Hakro's Ratings
new_user_ratings = [
    (0,6502, 5), #28 Days Later (2002)
    (0,48516,5), #Departed, The (2006)
    (0,48997,5), #Perfume: The Story of a Murderer (2006)
    (0,53000,4), #28 Weeks Later (2007)
    (0,58025,4), #Jumper (2008)
    (0,62374,5), #Body of Lies (2008)
    (0,64969,4), #Yes Man (2008)
    (0,66785,5), #Good, the Bad, the Weird, The (Joheunnom nabbeunnom isanghannom) (2008)
    (0,73808,4), #Chaser, The (Chugyeogja) (2008)
    (0,74750,3), #[REC]Â² (2009)
    (0,161131,4) #War Dogs (2016)
    ]

new_user_ratings_RDD = sc.parallelize(new_user_ratings)
print ('New user ratings: {}'.format(new_user_ratings_RDD.take(10)))

New user ratings: [(0, 6502, 5), (0, 48516, 5), (0, 48997, 5), (0, 53000, 4), (0, 58025, 4), (0, 62374, 5), (0, 64969, 4), (0, 66785, 5), (0, 73808, 4), (0, 74750, 3)]


Now we add them to the data we will use to train our recommender model. We use Spark's union() transformation for this.

In [12]:
complete_data_with_new_ratings_RDD = complete_ratings_data.union(new_user_ratings_RDD)


And finally we train the ALS model using all the parameters we selected before (when using the small dataset).


In [13]:
from time import time

t0 = time()
new_ratings_model = ALS.train(complete_data_with_new_ratings_RDD, best_rank, seed=seed,
                              iterations=iterations, lambda_=regularization_parameter)
tt = time() - t0

print ('New model trained in {} seconds'.format(round(tt,3)))

New model trained in 271.733 seconds


## Getting top recommendations
Let's now get some recommendations! For that we will get an RDD with all the movies the new user hasn't rated yet. We will them together with the model to predict ratings.

In [14]:
# new_user_ratings_ids = map(lambda x: x[1], new_user_ratings) # original version: get just movie IDs
new_user_ratings_ids = list(map(lambda x: x[1], new_user_ratings)) # fixed version: get just movie IDs
# keep just those not on the ID list (thanks Lei Li for spotting the error!)
new_user_unrated_movies_RDD = (complete_movies_data.filter(lambda x: x[0] not in new_user_ratings_ids).map(lambda x: (new_user_ID, x[0])))

# Use the input RDD, new_user_unrated_movies_RDD, with new_ratings_model.predictAll() to predict new ratings for the movies
new_user_recommendations_RDD = new_ratings_model.predictAll(new_user_unrated_movies_RDD)

We have our recommendations ready. Now we can print out the 25 movies with the highest predicted ratings. And join them with the movies RDD to get the titles, and ratings count in order to get movies with a minimum number of counts. First we will do the join and see what does the result looks like.

In [15]:
# Transform new_user_recommendations_RDD into pairs of the form (Movie ID, Predicted Rating)
new_user_recommendations_rating_RDD = new_user_recommendations_RDD.map(lambda x: (x.product, x.rating))
new_user_recommendations_rating_title_and_count_RDD = \
    new_user_recommendations_rating_RDD.join(complete_movies_titles).join(movie_rating_counts_RDD)
new_user_recommendations_rating_title_and_count_RDD.take(3)

[(257805, ((4.095148325957215, 'Best Sellers (2021)'), 32)),
 (154530, ((0.6543133078573464, 'Recto / verso'), 2)),
 (71910, ((3.4716985320338942, '"Tournament'), 268))]

So we need to flat this down a bit in order to have (Title, Rating, Ratings Count).


In [16]:
new_user_recommendations_rating_title_and_count_RDD = \
    new_user_recommendations_rating_title_and_count_RDD.map(lambda r: (r[1][0][1], r[1][0][0], r[1][1]))

Finally, get the highest rated recommendations for the new user, filtering out movies with less than 25 ratings.


In [17]:
top_movies = new_user_recommendations_rating_title_and_count_RDD.filter(lambda r: r[2]>=25).takeOrdered(25, key=lambda x: -x[1])

print ('TOP recommended movies (with more than 25 reviews):\n{}'.format('\n'.join(map(str, top_movies))))

TOP recommended movies (with more than 25 reviews):
('Planet Earth II (2016)', 5.284337575629234, 2041)
('Planet Earth (2006)', 5.272433660717352, 3015)
('Band of Brothers (2001)', 5.236756891795247, 2835)
('Twelve Angry Men (1954)', 5.211959224146703, 332)
('Blue Planet II (2017)', 5.192031367008683, 1267)
('Connections (1978)', 5.14816058851983, 57)
('Cosmos: A Spacetime Odissey', 5.146132076233016, 599)
('Cosmos', 5.144616785400825, 625)
('His Last Vow', 5.138349777810207, 41)
('"Shawshank Redemption', 5.134695749481148, 122296)
('The Blue Planet (2001)', 5.107675528524803, 1080)
('Legend of the Galactic Heroes: My Conquest Is the Sea of Stars (1988)', 5.103576575083804, 35)
('Human Planet (2011)', 5.066782098831175, 498)
('Firefly (2002)', 5.06446076954758, 895)
('The Mole (2020)', 5.0608015816239575, 46)
('Attack On Titan (2013)', 5.044420357209624, 263)
('Jimmy Carr: Live (2004)', 5.039070486228076, 43)
('Mushishi: The Shadow That Devours the Sun (2014)', 5.030936552966954, 33)
(

## Getting individual ratings
Another useful usecase is getting the predicted rating for a particular movie for a given user. The process is similar to the previous retreival of top recommendations but, instead of using predcitAll with every single movie the user hasn't rated yet, we will just pass the method a single entry with the movie we want to predict the rating for.

In [18]:
my_movie = sc.parallelize([(0, 500)]) # Quiz Show (1994)
# individual_movie_rating_RDD = new_ratings_model.predictAll(new_user_unrated_movies_RDD)
individual_movie_rating_RDD = new_ratings_model.predictAll(my_movie)
individual_movie_rating_RDD.take(1)

[Rating(user=0, product=500, rating=3.7440102980632575)]

## Tien Nguyen's rating

In [19]:
# The format of each line is (userID, movieID, rating)
# Tien Nguyen's rating
new_user_ratings = [
     (0,168366,3), # Beauty and the Beast (2017) 
     (0,166461,4), # Moana (2016) 
     (0,164909,5), # La La Land (2016) 
     (0,157296,5), # Finding Dory (2016) 
     (0,152081,5), # Zootopia (2016) 
     (0,130073,3), # Cinderella (2015) 
     (0,115667,4), # Love, Rosie (2014) 
     (0,111360,4), # Lucy (2014) 
     (0,106441,4), # Book Thief, The (2013) 
     (0,103335,4) # Despicable Me 2 (2013) 
    ]

new_user_ratings_RDD = sc.parallelize(new_user_ratings) 
print ('New user ratings: {}'.format(new_user_ratings_RDD.take(10)))

New user ratings: [(0, 168366, 3), (0, 166461, 4), (0, 164909, 5), (0, 157296, 5), (0, 152081, 5), (0, 130073, 3), (0, 115667, 4), (0, 111360, 4), (0, 106441, 4), (0, 103335, 4)]


In [20]:
complete_data_with_new_ratings_RDD = complete_ratings_data.union(new_user_ratings_RDD)

In [21]:
from time import time

t0 = time()
new_ratings_model = ALS.train(complete_data_with_new_ratings_RDD, best_rank, seed=seed,
                              iterations=iterations, lambda_=regularization_parameter)
tt = time() - t0

print ('New model trained in {} seconds'.format(round(tt,3)))

New model trained in 303.589 seconds


In [22]:
# new_user_ratings_ids = map(lambda x: x[1], new_user_ratings) # original version: get just movie IDs
new_user_ratings_ids = list(map(lambda x: x[1], new_user_ratings)) # fixed version: get just movie IDs
# keep just those not on the ID list (thanks Lei Li for spotting the error!)
new_user_unrated_movies_RDD = (complete_movies_data.filter(lambda x: x[0] not in new_user_ratings_ids).map(lambda x: (new_user_ID, x[0])))

# Use the input RDD, new_user_unrated_movies_RDD, with new_ratings_model.predictAll() to predict new ratings for the movies
new_user_recommendations_RDD = new_ratings_model.predictAll(new_user_unrated_movies_RDD)

In [23]:
#Transform new_user_recommendations_RDD into pairs of the form (Movie ID, Predicted Rating)
new_user_recommendations_rating_RDD = new_user_recommendations_RDD.map(lambda x: (x.product, x.rating))
new_user_recommendations_rating_title_and_count_RDD = \
    new_user_recommendations_rating_RDD.join(complete_movies_titles).join(movie_rating_counts_RDD)
new_user_recommendations_rating_title_and_count_RDD.take(3)

[(257805, ((3.979731755846204, 'Best Sellers (2021)'), 32)),
 (154530, ((0.6358890724081318, 'Recto / verso'), 2)),
 (71910, ((3.336709471864377, '"Tournament'), 268))]

In [24]:
new_user_recommendations_rating_title_and_count_RDD = \
    new_user_recommendations_rating_title_and_count_RDD.map(lambda r: (r[1][0][1], r[1][0][0], r[1][1]))

### User Tien Nguyen - Scenario 1 - FULL dataset, filtering out movies with less than 25 ratings (meaning 25 or more ratings)

In [25]:
# Sample Output of Top 15 Recommendations, filtering out movies with less than 25 ratings (meaning with 25 reviews or more):
top_movies = new_user_recommendations_rating_title_and_count_RDD.filter(lambda r: r[2]>=25).takeOrdered(15, key=lambda x: -x[1])

print ('TOP 15 recommended movies (with more than 25 reviews):\n{}'.format('\n'.join(map(str, top_movies))))

TOP 15 recommended movies (with more than 25 reviews):
('Journey to the Edge of the Universe (2008)', 5.223344488024097, 39)
('Planet Earth II (2016)', 5.219076137248911, 2041)
('Planet Earth (2006)', 5.172838735645503, 3015)
('Band of Brothers (2001)', 5.142512312131823, 2835)
('"Enemies of Reason', 5.135128185856803, 42)
('Blue Planet II (2017)', 5.113861833924004, 1267)
('The Mole (2020)', 5.099197931015478, 46)
('Cosmos: A Spacetime Odissey', 5.098613713727902, 599)
('Cosmos', 5.08508261172911, 625)
('Twelve Angry Men (1954)', 5.077455768493369, 332)
('Fight Club (1999)', 5.060543263731952, 86207)
('Pulp Fiction (1994)', 5.055097551135567, 108756)
('His Last Vow', 5.0371491161790845, 41)
('Death Note: Desu nôto (2006–2007)', 5.03580509279343, 509)
('Parasite (2019)', 5.023618406481615, 12399)


### User Tien Nguyen Scenario 2 - FULL dataset, filtering out movies with less than 100 ratings (meaning 100 or more ratings)

In [26]:
# Sample Output of Top 15 Recommendations, filtering out movies with less than 100 ratings (meaning with 50 reviews or more):
top_movies = new_user_recommendations_rating_title_and_count_RDD.filter(lambda r: r[2]>=100).takeOrdered(15, key=lambda x: -x[1])

print ('TOP 15 recommended movies (with more than 50 reviews):\n{}'.format('\n'.join(map(str, top_movies))))

TOP 15 recommended movies (with more than 50 reviews):
('Planet Earth II (2016)', 5.219076137248911, 2041)
('Planet Earth (2006)', 5.172838735645503, 3015)
('Band of Brothers (2001)', 5.142512312131823, 2835)
('Blue Planet II (2017)', 5.113861833924004, 1267)
('Cosmos: A Spacetime Odissey', 5.098613713727902, 599)
('Cosmos', 5.08508261172911, 625)
('Twelve Angry Men (1954)', 5.077455768493369, 332)
('Fight Club (1999)', 5.060543263731952, 86207)
('Pulp Fiction (1994)', 5.055097551135567, 108756)
('Death Note: Desu nôto (2006–2007)', 5.03580509279343, 509)
('Parasite (2019)', 5.023618406481615, 12399)
('"Shawshank Redemption', 4.998789585848811, 122296)
('Spider-Man: Across the Spider-Verse (2023)', 4.984472339237279, 528)
('The Blue Planet (2001)', 4.974525352812327, 1080)
('Attack On Titan (2013)', 4.959561023381621, 263)


In [27]:
my_movie = sc.parallelize([(0, 500)]) # Quiz Show (1994)
# individual_movie_rating_RDD = new_ratings_model.predictAll(new_user_unrated_movies_RDD)
individual_movie_rating_RDD = new_ratings_model.predictAll(my_movie)
individual_movie_rating_RDD.take(1)

[Rating(user=0, product=500, rating=3.606687880830948)]

## Hakro's rating

In [28]:
# The format of each line is (userID, movieID, rating)
# This is Tien Nguyen's rating
# new_user_ratings = [
#      (0,168366,3), # Beauty and the Beast (2017) 
#      (0,166461,4), # Moana (2016) 
#      (0,164909,5), # La La Land (2016) 
#      (0,157296,5), # Finding Dory (2016) 
#      (0,152081,5), # Zootopia (2016) 
#      (0,130073,3), # Cinderella (2015) 
#      (0,115667,4), # Love, Rosie (2014) 
#      (0,111360,4), # Lucy (2014) 
#      (0,106441,4), # Book Thief, The (2013) 
#      (0,103335,4) # Despicable Me 2 (2013) 
#     ]

# Hakro's Ratings
new_user_ratings = [
    (0,6502, 5), #28 Days Later (2002)
    (0,48516,5), #Departed, The (2006)
    (0,48997,5), #Perfume: The Story of a Murderer (2006)
    (0,53000,4), #28 Weeks Later (2007)
    (0,58025,4), #Jumper (2008)
    (0,62374,5), #Body of Lies (2008)
    (0,64969,4), #Yes Man (2008)
    (0,66785,5), #Good, the Bad, the Weird, The (Joheunnom nabbeunnom isanghannom) (2008)
    (0,73808,4), #Chaser, The (Chugyeogja) (2008)
    (0,74750,3), #[REC]Â² (2009)
    (0,161131,4) #War Dogs (2016)
    ]

new_user_ratings_RDD = sc.parallelize(new_user_ratings) 
print ('New user ratings: {}'.format(new_user_ratings_RDD.take(10)))

New user ratings: [(0, 6502, 5), (0, 48516, 5), (0, 48997, 5), (0, 53000, 4), (0, 58025, 4), (0, 62374, 5), (0, 64969, 4), (0, 66785, 5), (0, 73808, 4), (0, 74750, 3)]


In [29]:
complete_data_with_new_ratings_RDD = complete_ratings_data.union(new_user_ratings_RDD)

In [30]:
from time import time

t0 = time()
new_ratings_model = ALS.train(complete_data_with_new_ratings_RDD, best_rank, seed=seed,
                              iterations=iterations, lambda_=regularization_parameter)
tt = time() - t0

print ('New model trained in {} seconds'.format(round(tt,3)))

New model trained in 635.336 seconds


In [31]:
# new_user_ratings_ids = map(lambda x: x[1], new_user_ratings) # original version: get just movie IDs
new_user_ratings_ids = list(map(lambda x: x[1], new_user_ratings)) # fixed version: get just movie IDs
# keep just those not on the ID list (thanks Lei Li for spotting the error!)
new_user_unrated_movies_RDD = (complete_movies_data.filter(lambda x: x[0] not in new_user_ratings_ids).map(lambda x: (new_user_ID, x[0])))

# Use the input RDD, new_user_unrated_movies_RDD, with new_ratings_model.predictAll() to predict new ratings for the movies
new_user_recommendations_RDD = new_ratings_model.predictAll(new_user_unrated_movies_RDD)

In [32]:
#Transform new_user_recommendations_RDD into pairs of the form (Movie ID, Predicted Rating)
new_user_recommendations_rating_RDD = new_user_recommendations_RDD.map(lambda x: (x.product, x.rating))
new_user_recommendations_rating_title_and_count_RDD = \
    new_user_recommendations_rating_RDD.join(complete_movies_titles).join(movie_rating_counts_RDD)
new_user_recommendations_rating_title_and_count_RDD.take(3)

[(257805, ((4.095148325957215, 'Best Sellers (2021)'), 32)),
 (154530, ((0.6543133078573464, 'Recto / verso'), 2)),
 (71910, ((3.4716985320338942, '"Tournament'), 268))]

In [33]:
new_user_recommendations_rating_title_and_count_RDD = \
    new_user_recommendations_rating_title_and_count_RDD.map(lambda r: (r[1][0][1], r[1][0][0], r[1][1]))

### User Hakro - Scenario 1 - FULL dataset, filtering out movies with less than 25 ratings (meaning 25 or more ratings)

In [34]:
# Sample Output of Top 15 Recommendations, filtering out movies with less than 25 ratings (meaning with 25 reviews or more):
top_movies = new_user_recommendations_rating_title_and_count_RDD.filter(lambda r: r[2]>=25).takeOrdered(15, key=lambda x: -x[1])

print ('TOP 15 recommended movies (with more than 25 reviews):\n{}'.format('\n'.join(map(str, top_movies))))

TOP 15 recommended movies (with more than 25 reviews):
('Planet Earth II (2016)', 5.284337575629234, 2041)
('Planet Earth (2006)', 5.272433660717352, 3015)
('Band of Brothers (2001)', 5.236756891795247, 2835)
('Twelve Angry Men (1954)', 5.211959224146703, 332)
('Blue Planet II (2017)', 5.192031367008683, 1267)
('Connections (1978)', 5.14816058851983, 57)
('Cosmos: A Spacetime Odissey', 5.146132076233016, 599)
('Cosmos', 5.144616785400825, 625)
('His Last Vow', 5.138349777810207, 41)
('"Shawshank Redemption', 5.134695749481148, 122296)
('The Blue Planet (2001)', 5.107675528524803, 1080)
('Legend of the Galactic Heroes: My Conquest Is the Sea of Stars (1988)', 5.103576575083804, 35)
('Human Planet (2011)', 5.066782098831175, 498)
('Firefly (2002)', 5.06446076954758, 895)
('The Mole (2020)', 5.0608015816239575, 46)


### User Hakro - Scenario 2 - FULL dataset, filtering out movies with less than 100 ratings (meaning 100 or more ratings)

In [35]:
# Sample Output of Top 15 Recommendations, filtering out movies with less than 100 ratings (meaning with 100 reviews or more):
top_movies = new_user_recommendations_rating_title_and_count_RDD.filter(lambda r: r[2]>=100).takeOrdered(15, key=lambda x: -x[1])

print ('TOP 15 recommended movies (with more than 50 reviews):\n{}'.format('\n'.join(map(str, top_movies))))

TOP 15 recommended movies (with more than 50 reviews):
('Planet Earth II (2016)', 5.284337575629234, 2041)
('Planet Earth (2006)', 5.272433660717352, 3015)
('Band of Brothers (2001)', 5.236756891795247, 2835)
('Twelve Angry Men (1954)', 5.211959224146703, 332)
('Blue Planet II (2017)', 5.192031367008683, 1267)
('Cosmos: A Spacetime Odissey', 5.146132076233016, 599)
('Cosmos', 5.144616785400825, 625)
('"Shawshank Redemption', 5.134695749481148, 122296)
('The Blue Planet (2001)', 5.107675528524803, 1080)
('Human Planet (2011)', 5.066782098831175, 498)
('Firefly (2002)', 5.06446076954758, 895)
('Attack On Titan (2013)', 5.044420357209624, 263)
('"Usual Suspects', 5.022754137035191, 72893)
('The Adventures of Sherlock Holmes and Doctor Watson: The Hunt for the Tiger (1980)', 5.016035025070625, 125)
('Death Note: Desu nôto (2006–2007)', 5.013566097953532, 509)


In [36]:
my_movie = sc.parallelize([(0, 500)]) # Quiz Show (1994)
# individual_movie_rating_RDD = new_ratings_model.predictAll(new_user_unrated_movies_RDD)
individual_movie_rating_RDD = new_ratings_model.predictAll(my_movie)
individual_movie_rating_RDD.take(1)

[Rating(user=0, product=500, rating=3.7440102980632575)]

#### Interpret the results and provide Insights/Foresights on both scenarios

Combining the ratings and insights of Tien Nguyen and Hakro reveals a diverse range of preferences and high-quality recommendations. Tien Nguyen's ratings predominantly reflect a preference for modern animated and family-friendly movies such as "Finding Dory," "Zootopia," and "Moana." Tien also enjoys highly rated romantic dramas and action films, as indicated by "La La Land" and "Lucy." Meanwhile, Hakro's ratings highlight a taste for intense, thrilling, and dramatic movies, including "28 Days Later," "The Departed," and "Perfume: The Story of a Murderer." This user also appreciates action-packed and psychological films such as "War Dogs" and "Yes Man."

The recommendations for both users show a significant overlap in high-quality documentary series and classic films. In both scenarios, "Planet Earth II," "Band of Brothers," and "Cosmos: A Spacetime Odyssey" are top recommendations, emphasizing a shared appreciation for well-produced, educational content. The classic film "Twelve Angry Men" and popular movies like "Fight Club" and "Pulp Fiction" also appear prominently, indicating their broad appeal.

The ratings and recommendations reveal that both users have an eclectic mix of preferences, spanning various genres and styles. While Tien Nguyen leans more towards animated and family-friendly films, Hakro prefers intense thrillers and dramas. The combined recommendations suggest that both users would likely enjoy critically acclaimed documentaries, classic films, and a mix of popular mainstream movies. This diverse array of high-quality content ensures that both users will find films that align with their tastes while also exploring new genres and styles.